# NB06 — Model comparison: tables, Wilcoxon summary, figures (trains nothing)

Collects the 10-seed Wilcoxon results from NB06a/b/c, then builds the cross-dataset comparison, heatmap, ROC/PR curves, confusion matrices, Wilcoxon summary, CV summary and hyperparameter tables. Default models are the matched-schedule ones from NB10. All analysis cells are unchanged from the tested former NB06.

| | |
|---|---|
| **Required inputs** (Add Input → Your Work → Notebooks; always the newest version) | `01-preprocessing`, `02-coa-ann`, `03-coa-cnn`, `04-final-training`, `10-default-matched-schedule`, `06a-wilcoxon-amazon`, `06b-wilcoxon-yelp`, `06c-wilcoxon-twitter` |
| **Optional inputs** | `05b-cross-validation` (only for the CV summary table) |
| **Outputs** | `saved_results/{dataset}/{ann,cnn}_wilcoxon.json` (copied from 06a/b/c), `comparison.csv`, `saved_results/global/*` (tables and figures) |
| **Run time on Kaggle (estimate)** | minutes |
| **Accelerator / Internet** | none (CPU is enough) / off |

The training part of the former NB06 now lives in NB06a/b/c (one dataset each), so that every notebook fits one 9-hour session. Run NB06a/b/c and NB10 first.

Step-by-step instructions for the whole pipeline: `Code/Newest/RUN_GUIDE.md`. Background: `Decision/012-split-notebooks-for-9h-sessions.md`.

In [ ]:
import os, random as _random, warnings, re, string, json, gc, pickle
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from collections import Counter
warnings.filterwarnings("ignore")
os.environ["TF_CPP_MIN_LOG_LEVEL"] = "2"
SEED       = 42
MAX_VOCAB  = 50000
EMBED_DIM  = 300
DATASETS   = ["amazon", "yelp", "twitter"]

WORKING     = "/kaggle/working"
SHARED_DIR  = f"{WORKING}/shared_data"
RESULTS_DIR = f"{WORKING}/saved_results"
MODELS_DIR  = f"{WORKING}/saved_models"

def ensure_dirs(dataset=None):
    for base in [SHARED_DIR, RESULTS_DIR, MODELS_DIR]:
        os.makedirs(base, exist_ok=True)
        if dataset:
            os.makedirs(f"{base}/{dataset}", exist_ok=True)
        else:
            for ds in DATASETS:
                os.makedirs(f"{base}/{ds}", exist_ok=True)

def set_seeds(seed=SEED):
    os.environ["PYTHONHASHSEED"] = str(seed)
    _random.seed(seed)
    np.random.seed(seed)

set_seeds()
import tensorflow as tf
from tensorflow.keras.callbacks import EarlyStopping, ModelCheckpoint
from sklearn.metrics import (f1_score, accuracy_score, precision_score,
    recall_score, matthews_corrcoef, roc_auc_score, average_precision_score,
    classification_report, precision_recall_curve, confusion_matrix)
from sklearn.model_selection import StratifiedKFold
from sklearn.utils.class_weight import compute_class_weight
from scipy.stats import wilcoxon

tf.keras.mixed_precision.set_global_policy("mixed_float16")
strategy = tf.distribute.MirroredStrategy()
N_GPUS = strategy.num_replicas_in_sync
print(f"MirroredStrategy: {N_GPUS} GPU(s)")

ANN_BATCH = 1024 * N_GPUS
CNN_BATCH = 128  * N_GPUS
print(f"ANN_BATCH={ANN_BATCH}  CNN_BATCH={CNN_BATCH}")

def set_seeds_tf(seed=SEED):
    set_seeds(seed)
    tf.random.set_seed(seed)

set_seeds_tf()
print(f"TF {tf.__version__}")

ensure_dirs()


In [ ]:
def evaluate(name, y_true, y_pred, y_prob=None):
    auc = round(roc_auc_score(y_true, y_prob), 4) if y_prob is not None else float("nan")
    ap  = round(average_precision_score(y_true, y_prob), 4) if y_prob is not None else float("nan")
    rep = classification_report(y_true, y_pred, output_dict=True, zero_division=0)
    return {
        "Model"        : name,
        "Accuracy"     : round(accuracy_score(y_true, y_pred), 4),
        "Precision"    : round(precision_score(y_true, y_pred, zero_division=0), 4),
        "Recall"       : round(recall_score(y_true, y_pred, zero_division=0), 4),
        "F1-score"     : round(f1_score(y_true, y_pred, zero_division=0), 4),
        "Macro-F1"     : round(rep["macro avg"]["f1-score"], 4),
        "Weighted-F1"  : round(rep["weighted avg"]["f1-score"], 4),
        "MCC"          : round(matthews_corrcoef(y_true, y_pred), 4),
        "AUC-ROC"      : auc,
        "Avg-Precision": ap,
    }

def best_threshold(y_true, y_prob):
    prec, rec, thr = precision_recall_curve(y_true, y_prob)
    f1s = 2*prec*rec/(prec+rec+1e-9)
    idx = np.argmax(f1s[:-1])
    return float(thr[idx])

def make_es(patience=5, verbose=0):
    return EarlyStopping(monitor="val_loss", patience=patience,
                         restore_best_weights=True, verbose=verbose)

def make_tf_ds(X, y, batch, shuffle=True, seed=SEED):
    X = X.astype(np.float32)
    y = y.astype(np.float32)
    ds = tf.data.Dataset.from_tensor_slices((X, y))
    if shuffle: ds = ds.shuffle(min(len(y), 50000), seed=seed)
    return ds.batch(batch).cache().prefetch(tf.data.AUTOTUNE)

def make_seq_ds(X, y, batch, shuffle=True, seed=SEED):
    X = X.astype(np.int32)
    y = y.astype(np.float32)
    ds = tf.data.Dataset.from_tensor_slices((X, y))
    if shuffle: ds = ds.shuffle(min(len(y), 50000), seed=seed)
    return ds.batch(batch).cache().prefetch(tf.data.AUTOTUNE)

print("Helpers ready.")


## 1. Inputs

In [ ]:
# ── Inputs ────────────────────────────────────────────────────────────────────
U = "/kaggle/input/notebooks/abhaysharma01702"
NB01_DIR    = f"{U}/01-preprocessing"
NB01_SHARED = f"{NB01_DIR}/shared_data"
NB02_RESULTS = f"{U}/02-coa-ann/saved_results"
NB03_RESULTS = f"{U}/03-coa-cnn/saved_results"
NB04_RESULTS = f"{U}/04-final-training/saved_results"
NB05_RESULTS = f"{U}/05b-cross-validation/saved_results"      # optional (CV summary table only)
NB10_RESULTS = f"{U}/10-default-matched-schedule/saved_results"
WILCOXON_SLUGS = {"amazon": "06a-wilcoxon-amazon", "yelp": "06b-wilcoxon-yelp", "twitter": "06c-wilcoxon-twitter"}
for _name, _p in [("01-preprocessing", NB01_SHARED), ("02-coa-ann", NB02_RESULTS), ("03-coa-cnn", NB03_RESULTS),
                  ("04-final-training", NB04_RESULTS), ("10-default-matched-schedule", NB10_RESULTS),
                  ("05b-cross-validation (optional)", NB05_RESULTS)] + \
                 [(s, f"{U}/{s}/saved_results") for s in WILCOXON_SLUGS.values()]:
    print(f"  {_name:34s} {'✓' if os.path.isdir(_p) else '✗ not linked'}")


In [ ]:
import shutil
# ── Collect the Wilcoxon results of NB06a/b/c and choose the default models ────
STAT_SEEDS = [42, 7, 13, 99, 2024, 11, 23, 57, 101, 314]

# Matched-schedule defaults from NB10 (preferred). Falls back to NB01's 50/20-epoch defaults with a warning.
USE_MATCHED_DEFAULTS = all(os.path.exists(f"{NB10_RESULTS}/{ds}/default_matched_{a}_prob.npy")
                           for ds in DATASETS for a in ["ann", "cnn"])
print(f"Using {'NB10 matched-schedule' if USE_MATCHED_DEFAULTS else 'NB01 (50/20-epoch, SUPERSEDED)'} default models")
if not USE_MATCHED_DEFAULTS:
    print("WARNING: link 10-default-matched-schedule before producing paper figures.")

wilcoxon_all = {}
for ds, slug in WILCOXON_SLUGS.items():
    src = f"{U}/{slug}/saved_results/{ds}"
    if not all(os.path.exists(f"{src}/{mt}_wilcoxon.json") for mt in ["ann", "cnn"]):
        print(f"{ds}: {slug} not linked or not finished — skipped"); continue
    r = {}
    for mt in ["ann", "cnn"]:
        with open(f"{src}/{mt}_wilcoxon.json") as fh:
            r[mt] = json.load(fh)
    if r["ann"]["n_seeds"] < len(STAT_SEEDS):
        print(f"WARNING {ds}: only {r['ann']['n_seeds']}/{len(STAT_SEEDS)} seeds finished — do not report these p-values")
    wilcoxon_all[ds] = r
    dst = f"{RESULTS_DIR}/{ds}"; os.makedirs(dst, exist_ok=True)      # NB12 reads them from this notebook
    for fname in ["ann_wilcoxon.json", "cnn_wilcoxon.json", "wilcoxon.csv"]:
        if os.path.exists(f"{src}/{fname}"):
            shutil.copy(f"{src}/{fname}", f"{dst}/{fname}")
    for mt in ["ann", "cnn"]:
        s = r[mt]
        print(f"  {ds} {mt.upper()}: COA={s['coa_mean']}±{s['coa_std']}  Def={s['def_mean']}±{s['def_std']}  "
              f"wins={s['coa_wins']}/{s['n_seeds']}  p={s['p_value']:.4f}")


## 3. Load All Results for Comparison

In [ ]:
all_data = {}

for ds in DATASETS:
    d_dir = f"{NB01_SHARED}/{ds}"
    r_dir = f"{NB04_RESULTS}/{ds}"

    y_te         = np.load(f"{d_dir}/y_test.npy")
    lr_prob      = np.load(f"{d_dir}/lr_prob.npy")
    svm_scores   = np.load(f"{d_dir}/svm_scores.npy")
    if USE_MATCHED_DEFAULTS:   # 2026-10-12: same ≤100/30-epoch schedule as the NB04 COA models
        ann_def_prob = np.load(f"{NB10_RESULTS}/{ds}/default_matched_ann_prob.npy")
        cnn_def_prob = np.load(f"{NB10_RESULTS}/{ds}/default_matched_cnn_prob.npy")
    else:
        ann_def_prob = np.load(f"{d_dir}/ann_def_prob.npy")
        cnn_def_prob = np.load(f"{d_dir}/cnn_def_prob.npy")
    ann_coa_prob = np.load(f"{r_dir}/ann_coa_prob.npy")
    cnn_coa_prob = np.load(f"{r_dir}/cnn_coa_prob.npy")

    results = [
        evaluate("TF-IDF + LR",        y_te,(lr_prob>0.5).astype(int),      lr_prob),
        evaluate("TF-IDF + SVM",        y_te,(svm_scores>0).astype(int),     svm_scores),
        evaluate("Default ANN",         y_te,(ann_def_prob>0.5).astype(int),  ann_def_prob),
        evaluate("Default CNN+BiLSTM",  y_te,(cnn_def_prob>0.5).astype(int),  cnn_def_prob),
        evaluate("COA-ANN",             y_te,(ann_coa_prob>0.5).astype(int),  ann_coa_prob),
        evaluate("COA-CNN+BiLSTM",      y_te,(cnn_coa_prob>0.5).astype(int),  cnn_coa_prob),
    ]
    df = pd.DataFrame(results).sort_values("F1-score",ascending=False).reset_index(drop=True)
    df.index += 1
    df.to_csv(f"{RESULTS_DIR}/{ds}/comparison.csv")

    all_data[ds] = {
        "y_test" : y_te,
        "results_df": df,
        "probs"  : {"TF-IDF + LR":lr_prob,"TF-IDF + SVM":svm_scores,
                    "Default ANN":ann_def_prob,"Default CNN+BiLSTM":cnn_def_prob,
                    "COA-ANN":ann_coa_prob,"COA-CNN+BiLSTM":cnn_coa_prob},
        "preds"  : {"TF-IDF + LR":(lr_prob>0.5).astype(int),
                    "TF-IDF + SVM":(svm_scores>0).astype(int),
                    "Default ANN":(ann_def_prob>0.5).astype(int),
                    "Default CNN+BiLSTM":(cnn_def_prob>0.5).astype(int),
                    "COA-ANN":(ann_coa_prob>0.5).astype(int),
                    "COA-CNN+BiLSTM":(cnn_coa_prob>0.5).astype(int)},
    }

    print(f"\n{ds.upper()} — Best: {df.iloc[0]['Model']}  F1={df.iloc[0]['F1-score']}")
    print(df[["Model","F1-score","Macro-F1","MCC","AUC-ROC"]].to_string())

print("\nAll results loaded.")


## 4. Cross-Dataset Heatmap

In [ ]:
os.makedirs(f"{RESULTS_DIR}/global", exist_ok=True)

cross = []
for ds in DATASETS:
    for _,row in all_data[ds]["results_df"].iterrows():
        cross.append({"Dataset":ds.capitalize(),
                      **{k:row[k] for k in ["Model","F1-score","Macro-F1","MCC","AUC-ROC"]}})
cross_df = pd.DataFrame(cross)
cross_df.to_csv(f"{RESULTS_DIR}/global/cross_dataset_summary.csv",index=False)

pivot_f1  = cross_df.pivot_table(index="Model",columns="Dataset",values="F1-score",aggfunc="first").round(4)
pivot_mcc = cross_df.pivot_table(index="Model",columns="Dataset",values="MCC",aggfunc="first").round(4)

fig, axes = plt.subplots(1,2,figsize=(14,5))
sns.heatmap(pivot_f1,  annot=True,fmt=".4f",cmap="YlGnBu",linewidths=0.5,ax=axes[0],vmin=0.7,vmax=1.0)
axes[0].set_title("F1-score Heatmap")
sns.heatmap(pivot_mcc, annot=True,fmt=".4f",cmap="YlGnBu",linewidths=0.5,ax=axes[1],vmin=0.5,vmax=1.0)
axes[1].set_title("MCC Heatmap")
plt.suptitle("Cross-Dataset Performance — Per-Dataset COA Hyperparameters",fontsize=12)
plt.tight_layout()
plt.savefig(f"{RESULTS_DIR}/global/fig_cross_dataset_heatmap.pdf",dpi=300,bbox_inches="tight")
plt.show()
print("Saved: fig_cross_dataset_heatmap.pdf")


## 5. ROC + PR Curves

In [ ]:
from sklearn.metrics import roc_curve, average_precision_score
styles=["–","--","-.",":","-","--"]; styles=["-","--","-.",":","-","--"]
colors_roc=["#e74c3c","#e67e22","#3498db","#9b59b6","#1abc9c","#2c3e50"]
model_names=["TF-IDF + LR","TF-IDF + SVM","Default ANN",
             "Default CNN+BiLSTM","COA-ANN","COA-CNN+BiLSTM"]

fig, axes = plt.subplots(2,3,figsize=(20,12))
for col, ds in enumerate(DATASETS):
    d = all_data[ds]; y = d["y_test"]
    for mn,ls,col_c in zip(model_names,styles,colors_roc):
        fpr,tpr,_=roc_curve(y,d["probs"][mn])
        auc=roc_auc_score(y,d["probs"][mn])
        axes[0,col].plot(fpr,tpr,ls,color=col_c,lw=2,label=f"{mn} ({auc:.3f})")
    axes[0,col].plot([0,1],[0,1],"k--",lw=1,alpha=0.4)
    axes[0,col].set_title(f"ROC — {ds.capitalize()}")
    axes[0,col].set_xlabel("FPR"); axes[0,col].set_ylabel("TPR")
    axes[0,col].legend(fontsize=7,loc="lower right"); axes[0,col].grid(alpha=0.3)

    for mn,ls,col_c in zip(model_names,styles,colors_roc):
        prec,rec,_=precision_recall_curve(y,d["probs"][mn])
        ap=average_precision_score(y,d["probs"][mn])
        axes[1,col].plot(rec,prec,ls,color=col_c,lw=2,label=f"{mn} ({ap:.3f})")
    axes[1,col].axhline(y.sum()/len(y),color="grey",ls=":",lw=1.5)
    axes[1,col].set_title(f"PR — {ds.capitalize()}")
    axes[1,col].set_xlabel("Recall"); axes[1,col].set_ylabel("Precision")
    axes[1,col].legend(fontsize=7,loc="upper right"); axes[1,col].grid(alpha=0.3)

plt.suptitle("ROC & PR Curves — All Models × All Datasets",fontsize=14)
plt.tight_layout()
plt.savefig(f"{RESULTS_DIR}/global/fig_roc_pr.pdf",dpi=300,bbox_inches="tight")
plt.show()
print("Saved: fig_roc_pr.pdf")


## 6. Confusion Matrices (COA Models)

In [ ]:
fig, axes = plt.subplots(2,3,figsize=(18,11))
for col, ds in enumerate(DATASETS):
    d=all_data[ds]; y=d["y_test"]
    for row, mn in enumerate(["COA-ANN","COA-CNN+BiLSTM"]):
        cm=confusion_matrix(y,d["preds"][mn])
        f1=f1_score(y,d["preds"][mn]); mcc=matthews_corrcoef(y,d["preds"][mn])
        sns.heatmap(cm,annot=True,fmt="d",cmap="Blues",ax=axes[row,col],
                    xticklabels=["Neg","Pos"],yticklabels=["Neg","Pos"],linewidths=0.5)
        axes[row,col].set_title(f"{mn}\n{ds.capitalize()} | F1={f1:.4f} MCC={mcc:.4f}",fontsize=9)
        axes[row,col].set_xlabel("Predicted"); axes[row,col].set_ylabel("Actual")
plt.suptitle("Confusion Matrices — COA Models × All Datasets",fontsize=13)
plt.tight_layout()
plt.savefig(f"{RESULTS_DIR}/global/fig_confusion_matrices.pdf",dpi=300,bbox_inches="tight")
plt.show()


## 7. Statistical Significance Summary

In [ ]:
rows=[]
for ds in [x for x in DATASETS if x in wilcoxon_all]:
    for mt,label in [("ann","COA-ANN vs Default ANN"),("cnn","COA-CNN+BiLSTM vs Default CNN+BiLSTM")]:
        s=wilcoxon_all[ds][mt]
        rows.append({"Dataset":ds.capitalize(),"Comparison":label,
                     f"COA F1":f"{s['coa_mean']}±{s['coa_std']}",
                     f"Def F1":f"{s['def_mean']}±{s['def_std']}",
                     "n seeds":s["n_seeds"],"p-value":s["p_value"],"Significant":"YES ✓" if s["significant"] else "NO ✗"})

stat_df=pd.DataFrame(rows)
if stat_df.empty:   # 2026-10-12: time limit reached before any dataset had >=2 finished seeds
    print("No Wilcoxon results yet — rerun with RESUME_SLUGS to finish the seeds.")
else:
    print("=== Wilcoxon Summary ===\n")
    print(stat_df.to_string(index=False))
    stat_df.to_csv(f"{RESULTS_DIR}/global/wilcoxon_summary.csv",index=False)

    fig,ax=plt.subplots(figsize=(13,4))
    labels=[f"{r['Dataset']}\n{r['Comparison'].split(' vs ')[0]}" for _,r in stat_df.iterrows()]
    pvals=stat_df["p-value"].values
    ax.bar(range(len(labels)),pvals,color=["#2ecc71" if p<0.05 else "#e74c3c" for p in pvals],alpha=0.85)
    ax.axhline(0.05,color="black",ls="--",lw=2,label="α=0.05")
    ax.set_xticks(range(len(labels))); ax.set_xticklabels(labels,rotation=15,ha="right",fontsize=9)
    ax.set_ylabel("p-value"); ax.set_title("Wilcoxon p-values (green=significant, red=not)")
    ax.legend(); ax.grid(axis="y",alpha=0.3)
    plt.tight_layout()
    plt.savefig(f"{RESULTS_DIR}/global/fig_wilcoxon_pvalues.pdf",dpi=300,bbox_inches="tight")
    plt.show()


## 8. CV Summary

In [ ]:
cv_rows=[]
for ds in DATASETS:
    for mt in ["ann","cnn"]:
        try:
            s=json.load(open(f"{NB05_RESULTS}/{ds}/{mt}_cv_summary.json"))
            sm=s["summary"]
            cv_rows.append({"Dataset":ds.capitalize(),"Model":f"COA-{mt.upper()}",
                "F1":f"{sm['f1']['mean']}±{sm['f1']['std']}",
                "MCC":f"{sm['mcc']['mean']}±{sm['mcc']['std']}",
                "AUC":f"{sm['auc']['mean']}±{sm['auc']['std']}"})
        except: print(f"Missing {mt}_cv_summary.json for {ds}")
cv_df=pd.DataFrame(cv_rows)
print("=== 5-Fold CV Summary ===\n")
print(cv_df.to_string(index=False))
cv_df.to_csv(f"{RESULTS_DIR}/global/cv_summary.csv",index=False)


## 9. COA Hyperparameters per Dataset

In [ ]:
print("=== COA Hyperparameters (per dataset) ===\n")
hparam_rows=[]
for ds in DATASETS:
    ann=json.load(open(f"{NB02_RESULTS}/{ds}/coa_ann.json"))
    cnn=json.load(open(f"{NB03_RESULTS}/{ds}/coa_cnn.json"))
    hparam_rows.append({
        "Dataset":ds.capitalize(),
        "ANN H1":ann["best_h1"],"ANN H2":ann["best_h2"],"ANN LR":round(ann["best_lr"],8),
        "ANN val_F1":ann["best_val_f1"],
        "CNN Filters":cnn["best_filters"],"CNN LSTM":cnn["best_lstm_units"],
        "CNN LR":round(cnn["best_lr"],8),"CNN val_F1":cnn["best_val_f1"],
    })
hp_df=pd.DataFrame(hparam_rows)
print(hp_df.to_string(index=False))
hp_df.to_csv(f"{RESULTS_DIR}/global/coa_hyperparameters.csv",index=False)


## 10. Final Summary

In [ ]:
print("="*70)
print("FINAL RESULTS — COA-Optimised Sentiment Analysis")
print("3 Datasets | Per-Dataset COA | All Issues Fixed")
print("="*70)

for ds in DATASETS:
    df=all_data[ds]["results_df"]
    best=df.iloc[0]
    print(f"\n{ds.upper()} — Best: {best['Model']}")
    for col in ["F1-score","Macro-F1","MCC","AUC-ROC","Avg-Precision"]:
        print(f"  {col:16s}: {best[col]:.4f}")

print("\nAll figures and tables:")
for f_ in sorted(os.listdir(f"{RESULTS_DIR}/global")):
    print(f"  ✓ {f_}")
